# Held-out network-only freeze checkpoint

This notebook reads **compact tracked CSV/JSON and source files only**. It does not open network miniSEED, full score arrays, a catalog, a family-label table, or any DAS HDF5.

**Recorded outcome:** all 12 registered hours passed fixed network QC. The primary historical-template branch retained 12 candidates and the auxiliary generic branch retained 21. The frozen interval-scoped union has 33 rows because the branches have zero matches within the registered 8-second window. These are unadjudicated network candidates—not earthquakes, repeaters, or DAS catalog extensions.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
matches = [
    path for path in [ROOT, *ROOT.parents]
    if (path / 'outputs' / 'heldout_v2' / 'network').is_dir()
]
if not matches:
    raise FileNotFoundError('Run from repeaters_v2 or one of its subdirectories')
PROJECT = matches[0]
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.network_union import build_time_only_union

NETWORK = PROJECT / 'outputs' / 'heldout_v2' / 'network'
REGISTRATION = PROJECT / 'outputs' / 'heldout_v2' / 'registration'
CONFIG = PROJECT / 'config'
pd.set_option('display.max_columns', 50)
print('Project:', PROJECT)
print('Inputs: compact frozen products only; no catalog or waveform is opened.')

## 1. Frozen provenance and access ledger

These assertions fail if a registered threshold, released runner source, aggregate candidate table, or time-only union changes. The SLURM ledger is execution provenance; the scientific freeze is the checksummed candidate-generation status.

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

with (CONFIG / 'heldout_network_validation.json').open(encoding='utf-8') as handle:
    config = json.load(handle)
with (REGISTRATION / 'network_runner_release.json').open(encoding='utf-8') as handle:
    release = json.load(handle)
with (NETWORK / 'candidate_generation_status.json').open(encoding='utf-8') as handle:
    status = json.load(handle)
interval_status = pd.read_csv(NETWORK / 'interval_status.csv')
templates = pd.read_csv(NETWORK / 'template_candidates_time_only.csv')
generics = pd.read_csv(NETWORK / 'generic_candidates_time_only.csv')
union = pd.read_csv(NETWORK / 'network_union_time_only.csv')
sources = pd.read_csv(NETWORK / 'source_request_ledger.csv')
slurm = pd.read_csv(NETWORK / 'slurm_execution.csv')

assert sha256(CONFIG / 'heldout_network_validation.json') == status['heldout_network_config_sha256']
assert sha256(REGISTRATION / 'network_runner_release.json') == status['network_runner_release_sha256']
assert release['runner_commit_sha'] == status['runner_commit_sha']
assert release['runner_commit_sha'] == release['remote_branch_sha']
assert release['remote_branch_sha_verified_equal']
for relative, expected_hash in release['runner_implementation_files'].items():
    assert sha256(PROJECT / relative) == expected_hash
product_hashes = {
    'source_request_ledger.csv': 'source_request_ledger_sha256',
    'trace_qc.csv': 'trace_QC_sha256',
    'interval_status.csv': 'interval_status_sha256',
    'template_candidates_time_only.csv': 'template_candidate_sha256',
    'generic_candidates_time_only.csv': 'generic_candidate_sha256',
    'network_union_time_only.csv': 'time_only_union_sha256',
}
for filename, field in product_hashes.items():
    assert sha256(NETWORK / filename) == status[field]
assert status['status'] == 'PASS'
assert status['network_union_complete_and_frozen']
assert status['all_interval_candidate_tables_checksums_verified_before_union']
assert not status['threshold_recalibration_performed']
assert not status['candidate_deletion_after_review_performed']
assert not status['cross_interval_candidate_matching_performed']
assert status['interval_count'] == len(interval_status) == len(slurm) == 12
assert (interval_status['interval_status'] == 'PASS').all()
assert (slurm['state'] == 'COMPLETED').all()
assert (slurm['exit_code'] == '0:0').all()
assert len(templates) == status['template_candidate_count'] == 12
assert len(generics) == status['generic_candidate_count'] == 21
assert len(union) == status['time_only_union_candidate_count'] == 33
assert (union['catalog_fields_used_in_grouping'] == 0).all()
assert (union['family_assignment'] == 'not_assigned').all()

access = {
    'held-out network source files used': status['heldout_network_waveform_files_opened'],
    'held-out catalog rows opened': status['heldout_catalog_event_rows_opened'],
    'held-out DAS HDF5 files opened': status['heldout_DAS_HDF5_files_opened'],
    'held-out DAS datasets opened': status['heldout_DAS_HDF5_datasets_opened'],
    'held-out family-label rows opened': status['heldout_family_label_rows_opened'],
    'family assignments made': status['candidate_family_assignments_made'],
}
display(pd.Series({
    'freeze status': status['status'],
    'registered hours': status['heldout_total_duration_h'],
    'intervals passing fixed QC': status['interval_generation_status_counts']['PASS'],
    'available network sources': f"{status['available_source_count']}/{status['source_request_row_count']}",
    'usable traces': f"{status['usable_trace_count']}/{status['trace_QC_row_count']}",
    'template candidates': len(templates),
    'generic candidates': len(generics),
    'cross-branch pairs at 8 s': status['cross_branch_pair_count'],
    'frozen union rows': len(union),
    'runner/remote SHA': release['runner_commit_sha'],
    'SLURM array job': int(slurm['array_job_id'].iloc[0]),
}, name='network-only checkpoint').to_frame())
display(pd.Series(access, name='count').to_frame())

## 2. What the network-only scan produced

Candidate counts are strongly concentrated by hour and branch. The one unavailable source is retained explicitly. A `PASS` here means the registered mechanics and QC executed; it does **not** validate any candidate as a local earthquake or repeater.

In [ ]:
interval_ids = status['interval_ids']
counts = pd.DataFrame(index=interval_ids)
counts.index.name = 'interval_id'
counts['template'] = templates.groupby('interval_id').size().reindex(interval_ids, fill_value=0)
counts['generic'] = generics.groupby('interval_id').size().reindex(interval_ids, fill_value=0)
counts['union'] = union.groupby('interval_id').size().reindex(interval_ids, fill_value=0)
counts = counts.join(
    interval_status.set_index('interval_id')[[
        'available_source_count', 'usable_component_count', 'usable_station_count'
    ]]
)
assert (counts['union'] == counts['template'] + counts['generic']).all()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
counts[['template', 'generic']].plot.bar(
    stacked=True, ax=axes[0], color=['tab:blue', 'tab:orange']
)
axes[0].set(ylabel='frozen candidate count', xlabel='', title='Candidates by held-out hour')
axes[0].tick_params(axis='x', rotation=45)
axes[1].scatter(
    templates['bank_score'],
    templates['station_support_count_at_0p2'],
    label='template bank', color='tab:blue', alpha=0.8,
)
axes[1].scatter(
    generics['coincidence_score'],
    generics['station_support_count_at_declared_ratio'],
    label='generic trigger', color='tab:orange', alpha=0.8,
)
axes[1].set(
    xlabel='branch score (not comparable across branches)',
    ylabel='station support at branch recording level',
    title='Frozen candidate score/support',
)
axes[1].legend()
plt.show()

display(counts)
missing = sources.loc[sources['status'] != 'available', [
    'interval_id', 'source_name', 'network', 'status', 'error'
]]
display(missing if len(missing) else pd.DataFrame({'status': ['all sources available']}))

## 3. Advisor sandbox: display-only branch timing

`EXPLORATORY_MATCH_WINDOW_S` and `DISPLAY_INTERVAL` change only this in-memory display. They do not overwrite the registered 8-second union and cannot be used to repair the held-out detector. Template times estimate origins; generic times are array arrivals, so very broad pairing windows are not physically defensible.

In [ ]:
EXPLORATORY_MATCH_WINDOW_S = 8.0
DISPLAY_INTERVAL = 'all'  # or, for example, 'heldout_03'


def rebuilt_union(window_s):
    rows = []
    for interval_id in interval_ids:
        local = build_time_only_union(
            templates.loc[templates['interval_id'] == interval_id].to_dict('records'),
            generics.loc[generics['interval_id'] == interval_id].to_dict('records'),
            maximum_difference_s=float(window_s),
            identifier_prefix=f'exploratory_{interval_id}',
        )
        rows.extend({'interval_id': interval_id, **row} for row in local)
    return pd.DataFrame(rows)

sandbox_union = rebuilt_union(EXPLORATORY_MATCH_WINDOW_S)
assert sha256(NETWORK / 'network_union_time_only.csv') == status['time_only_union_sha256']
window_grid = [2, 4, 8, 12, 30, 60, 120, 300]
sensitivity = []
for window_s in window_grid:
    rebuilt = rebuilt_union(window_s)
    sensitivity.append({
        'exploratory_window_s': window_s,
        'cross_branch_pairs': int((rebuilt['branch_count'] == 2).sum()),
        'union_rows': len(rebuilt),
    })
sensitivity = pd.DataFrame(sensitivity)
assert sensitivity.loc[sensitivity['exploratory_window_s'] == 8, 'cross_branch_pairs'].iloc[0] == 0

minimum_rows = []
for interval_id in interval_ids:
    t = templates.loc[templates['interval_id'] == interval_id, 'origin_epoch_s'].to_numpy()
    g = generics.loc[generics['interval_id'] == interval_id, 'trigger_epoch_s'].to_numpy()
    differences = np.abs(t[:, None] - g[None, :]) if len(t) and len(g) else np.empty((0, 0))
    minimum_rows.append({
        'interval_id': interval_id,
        'minimum_cross_branch_absolute_difference_s': (
            float(differences.min()) if differences.size else np.nan
        ),
    })
minimum = pd.DataFrame(minimum_rows)

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)
ax.plot(sensitivity['exploratory_window_s'], sensitivity['cross_branch_pairs'], marker='o')
ax.axvline(8, color='black', linestyle='--', label='frozen window')
ax.set(xlabel='exploratory window (s)', ylabel='cross-branch pairs', title='Display-only timing sensitivity')
ax.legend()
plt.show()
display(sensitivity)
display(minimum.dropna())
view = union if DISPLAY_INTERVAL == 'all' else union.loc[union['interval_id'] == DISPLAY_INTERVAL]
display(view.sort_values(['interval_id', 'representative_epoch_s']))

## 4. What this changes—and what it does not

The best-network comparator is finally materialized on the same 12 hours that will be used for DAS. That directly answers the methodological objection “why not find them on the seismic network first?” But the 33 rows are still raw candidates. The next stage may attach catalog evidence only to this immutable union; it may not delete rows or repair thresholds. Held-out DAS then runs from the registered intervals independently of these network times.

In [ ]:
protocol = pd.DataFrame([
    {'order': 1, 'stage': 'Push remote-verified network runner', 'status': 'complete'},
    {'order': 2, 'stage': 'Run all 12 network-only intervals', 'status': 'complete'},
    {'order': 3, 'stage': 'Checksum/freeze time-only network union', 'status': 'complete'},
    {'order': 4, 'stage': 'Register and run catalog audit on immutable union', 'status': 'next'},
    {'order': 5, 'stage': 'Register/run frozen DAS-v2 independently', 'status': 'pending'},
    {'order': 6, 'stage': 'Compare unique events and interval uncertainty', 'status': 'pending'},
])
display(protocol.set_index('order'))
print('Catalog gate:', status['catalog_access_gate'])
print('DAS gate:', status['DAS_access_gate'])
print('Family gate:', status['family_assignment_gate'])
print('Scientific claim remains STOP: no candidate has been adjudicated.')